# RoboBarista Vision – FAIRINO FR5, 3 RGB-D-Kameras

Das Notebook installiert selbst die Software und lädt den Programmcode aus deinem GitHub-Repository. **Kein ZIP-Upload nötig.**

**Status:** erste Integration noch nicht mit MuJoCo gerendert; lokale Pose-/Marker-Tests bestanden. Es ist möglich, dass die erste Visualisierung Kalibrierungs- oder Asset-Fixes benötigt.

**Ziel:** Die aus RGB-/Tiefenbildern geschätzten Objektposen getrennt von MuJoCo-Ground-Truth messen. Ein Test besteht nur, wenn jeder Tag in mindestens zwei Kameras und die Fehler unter den vorläufigen Schwellen liegen.


In [ ]:
# Installation (nur beim ersten Start einer neuen Colab-Laufzeit)
%pip -q install 'numpy>=1.26,<3' 'scipy>=1.11' 'opencv-contrib-python-headless>=4.8,<5' 'mujoco>=3.2,<4' 'Pillow>=10' 'matplotlib>=3.8'


In [ ]:
# Repository laden – kein Dateiupload
from pathlib import Path
import subprocess, sys, os
PROJECT = Path('/content/robobarista')
if not PROJECT.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Coffee-Tobi/robocasa.git',str(PROJECT)],check=True)
else:
    subprocess.run(['git','-C',str(PROJECT),'pull','--ff-only'],check=True)
print('Projekt:',PROJECT)
subprocess.run([sys.executable,str(PROJECT/'camera_benchmark.py'),'--self-test'],cwd=PROJECT,check=True)


In [ ]:
# Simulierte RGB-D-Kameras und Positionsfehler messen
from pathlib import Path
import subprocess, os, sys
cmd=[sys.executable,'camera_benchmark.py','--trials','3','--width','960','--height','720','--min-cameras','2','--max-mm','5','--max-deg','5']
result=subprocess.run(cmd,cwd=str(PROJECT),env={**os.environ,'MUJOCO_GL':'egl'})
print('Benchmark exit code:',result.returncode, '(0=bestanden, 2=Genauigkeit nicht erreicht)')
if result.returncode not in (0,2):
    print('Integrationsfehler: Bitte den vollständigen Traceback teilen.')


In [ ]:
# Ergebnisse und drei RGB-D-Ansichten anzeigen
import json
from IPython.display import display, Image
results=PROJECT/'runs'/'first_benchmark'
p=results/'rgbd_montage.jpg'
if p.exists(): display(Image(filename=str(p)))
p=results/'summary.json'
if p.exists(): print(json.dumps(json.loads(p.read_text()),indent=2))
print('Weiteres:',results)


**Wichtig:** Es werden bislang vereinfachte und feststehende Kännchen-/Tassenmodelle verwendet. Der FR5 selbst ist ein bestehendes MuJoCo-Modell. Die Bewegung und das Erlernen einer Latte-Art-Trajektorie kommen erst später.

Schicke für die erste Auswertung bitte `runs/first_benchmark/rgbd_montage.jpg`, `summary.json` und gegebenenfalls die Fehlermeldung. Ein Zielwert von 5 mm / 5° ist *kein* bereits erreichter Messwert.
